In [1]:
import pandas as pd 
import numpy as np

df = pd.read_csv('3d_metal_data.csv')
df.describe()

,lat,sws0,B0,M_s,T_C,Pugh,Cauchy,c11,c12,c44
count,472.000000,472.000000,472.000000,472.000000,472.000000,472.000000,472.000000,472.000000,472.000000,472.000000
mean,0.258475,2.773790,128.795929,0.175132,296.363643,2.490623,24.137836,164.605988,110.890900,86.753064
std,0.438261,0.078438,28.249685,0.170995,325.349667,0.363424,15.794194,45.511373,23.804906,19.176130
min,0.000000,2.627565,53.520951,0.000000,0.000000,0.895650,-62.483518,74.009968,27.901137,41.701630
25%,0.000000,2.714655,109.503477,0.000859,0.062141,2.295952,15.148457,134.013839,94.451072,72.870409
50%,0.000000,2.764804,129.020781,0.149307,177.170037,2.508167,25.354799,157.861832,112.997533,84.732792
75%,1.000000,2.820491,147.262917,0.286727,512.506954,2.697268,32.555362,185.341833,126.774720,99.394439
max,1.000000,3.078630,208.267032,0.702256,1540.174280,5.799876,86.867106,380.786078,188.318444,151.583784


In [2]:
df_unique = df.drop_duplicates(subset=['Formula'], keep='first')
df_unique.describe()

,lat,sws0,B0,M_s,T_C,Pugh,Cauchy,c11,c12,c44
count,450.000000,450.000000,450.000000,450.000000,450.000000,450.000000,450.000000,450.000000,450.000000,450.000000
mean,0.255556,2.775660,128.687417,0.168501,287.792845,2.488199,24.244509,164.860800,110.600725,86.356216
std,0.436658,0.078938,28.516468,0.168013,324.041257,0.364621,16.000243,46.258553,23.871802,19.147239
min,0.000000,2.627565,53.520951,0.000000,0.000000,0.895650,-62.483518,74.009968,27.901137,41.701630
25%,0.000000,2.716447,108.871761,0.000645,0.036040,2.286176,15.093068,133.680827,94.045719,72.623352
50%,0.000000,2.768913,128.582186,0.143255,167.177611,2.507085,25.313352,158.079699,112.226728,84.240233
75%,1.000000,2.822078,147.557355,0.281225,498.232661,2.695472,33.584177,186.253420,126.750634,98.955731
max,1.000000,3.078630,208.267032,0.702256,1540.174280,5.799876,86.867106,380.786078,188.318444,151.583784


In [5]:
# df = df_unique
properties = ['M_s', 'T_C', 'Pugh', 'Cauchy']

df = df_unique.copy()[['Formula', 'lat']+properties]

# min_values = {col: df[col].quantile(0.1) for col in properties}
# min_values = {col: df[col].min() for col in columns}
# df = df[~df.apply(lambda row: any(row[col] <= min_values[col] for col in properties), axis=1)]
# display(df.describe())

# ## Optional: remove max
# max_values = {col: df[col].max() for col in ['Pugh', 'Cauchy']}
# # min_values = {col: df[col].min() for col in columns}
# df = df[~df.apply(lambda row: any(row[col] >= max_values[col] for col in ['Pugh', 'Cauchy']), axis=1)]
# display(df.describe())
from sklearn.preprocessing import StandardScaler, MinMaxScaler
scaler_y = StandardScaler() 
scaler_y.fit(df[properties])
normalized_props = [f'norm_{i}' for i in properties]
df[normalized_props] = scaler_y.transform(df[properties])
df.describe()
# df[normalized_props] = df[properties].copy().apply(
#     lambda x: (x - x.min()) / (x.max() - x.min())
# )


# df['pareto'] = np.linalg.norm(df[normalized_props], axis=1)
# df.to_csv('Pareto_data.csv', index=False)
# df.describe().to_csv('Pareto_statistics.csv')
# display(df.describe())

,lat,M_s,T_C,Pugh,Cauchy,norm_M_s,norm_T_C,norm_Pugh,norm_Cauchy
count,450.000000,450.000000,450.000000,450.000000,450.000000,4.500000e+02,4.500000e+02,4.500000e+02,4.500000e+02
mean,0.255556,0.168501,287.792845,2.488199,24.244509,-6.315935e-17,-4.736952e-17,-9.276530e-16,7.894919e-17
std,0.436658,0.168013,324.041257,0.364621,16.000243,1.001113e+00,1.001113e+00,1.001113e+00,1.001113e+00
min,0.000000,0.000000,0.000000,0.895650,-62.483518,-1.004019e+00,-8.891249e-01,-4.372544e+00,-5.426452e+00
25%,0.000000,0.000645,0.036040,2.286176,15.093068,-1.000175e+00,-8.890136e-01,-5.546810e-01,-5.725929e-01
50%,0.000000,0.143255,167.177611,2.507085,25.313352,-1.504306e-01,-3.726361e-01,5.185324e-02,6.687604e-02
75%,1.000000,0.281225,498.232661,2.695472,33.584177,6.716716e-01,6.501457e-01,5.690935e-01,5.843701e-01
max,1.000000,0.702256,1540.174280,5.799876,86.867106,3.180396e+00,3.869184e+00,9.092627e+00,3.918209e+00


In [6]:
current_lb = np.min(df[normalized_props], axis=0)
current_lb

norm_M_s      -1.004019
norm_T_C      -0.889125
norm_Pugh     -4.372544
norm_Cauchy   -5.426452
dtype: float64

In [12]:
(df[normalized_props]-current_lb).describe()

,norm_M_s,norm_T_C,norm_Pugh,norm_Cauchy
count,450.000000,450.000000,450.000000,450.000000
mean,1.004019,0.889125,4.372544,5.426452
std,1.001113,1.001113,1.001113,1.001113
min,0.000000,0.000000,0.000000,0.000000
25%,0.003844,0.000111,3.817863,4.853859
50%,0.853588,0.516489,4.424398,5.493328
75%,1.675690,1.539271,4.941638,6.010822
max,4.184414,4.758309,13.465172,9.344661


In [10]:
df['Pareto'] = np.linalg.norm(df[normalized_props]-current_lb, axis=1)
df['Pareto'].describe()

count    450.000000
mean       7.279864
std        1.170794
min        1.481305
25%        6.749029
50%        7.439447
75%        8.040165
max       14.367609
Name: Pareto, dtype: float64

In [14]:
sorted_df = df.sort_values(by=['Pareto'],ascending=False).reset_index(drop=True)
sorted_df.head(20).to_csv('Pareto_optimal_candidates.csv', index=False)
sorted_df.head(20)

,Formula,lat,M_s,T_C,Pugh,Cauchy,norm_M_s,norm_T_C,norm_Pugh,norm_Cauchy,Pareto
0,Sc0.191Ti0.172V0.171Cr0.233Zn0.233,1,0.000000e+00,0.000000,5.799876,17.616086,-1.004019,-0.889125,9.092627,-0.414731,14.367609
1,Cr0.055Fe0.262Ni0.184Cu0.242Zn0.257,1,2.184742e-01,469.086455,3.973863,86.867106,0.297767,0.560100,4.079077,3.918209,12.749411
2,Ti0.113V0.123Cr0.180Co0.084Zn0.500,0,4.075883e-03,99.736301,3.185066,67.276411,-0.979732,-0.580993,1.913335,2.692446,10.272503
3,Fe0.359Co0.362Ni0.097Cu0.094Zn0.088,0,5.956033e-01,1540.174280,2.465197,31.800317,2.544904,3.869184,-0.063155,0.472756,9.413175
4,Sc0.214Ti0.082V0.333Cr0.327Zn0.044,0,2.498030e-07,0.000000,2.923076,56.875252,-1.004017,-0.889125,1.194009,2.041660,9.314463
5,Ti0.208Co0.525Ni0.110Cu0.116Zn0.041,0,2.368283e-01,629.296523,2.964990,46.274678,0.407131,1.055062,1.309090,1.378397,9.184669
6,Mn0.039Fe0.427Co0.074Ni0.067Cu0.393,0,5.073355e-01,1149.907290,2.781018,30.341888,2.018957,2.663469,0.803970,0.381505,9.071272
7,Ti0.230Co0.284Ni0.102Cu0.354Zn0.030,1,7.513873e-02,140.454488,3.183500,41.227041,-0.556302,-0.455196,1.909035,1.062574,9.052869
8,Sc0.067Ti0.051Ni0.349Cu0.113Zn0.420,1,5.148208e-08,0.000000,3.273654,36.341693,-1.004018,-0.889125,2.156565,0.756904,8.992395
9,Sc0.115Fe0.088Co0.564Ni0.181Zn0.052,1,3.895318e-01,860.918967,2.891577,35.604252,1.317019,1.770651,1.107525,0.710763,8.953111
